# ForgeLM — Synthetic Data Pipeline

Generate training data from a teacher model, then fine-tune a student model on it.

Supports 3 teacher backends:
- **API** — OpenAI, Anthropic, or any OpenAI-compatible endpoint
- **Local** — Load a HuggingFace model for generation
- **File** — Use pre-generated responses (offline/reproducible)

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/synthetic_data_training.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
# Pinned to v0.5.0; bump on each release
!pip install -q --no-cache-dir 'forgelm[qlora]==0.5.0' bitsandbytes
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU detected.")
    print("Data generation works without GPU (API/file backend).")
    print("Training requires GPU — Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create seed prompts and pre-generated responses
# For this demo, we use the "file" backend (no API key needed)
# In production, you'd use "api" with GPT-4/Claude or "local" with a large model
import json

# Simulate teacher responses (pre-generated)
teacher_data = [
    {
        "prompt": "What is machine learning?",
        "response": "Machine learning is a subset of artificial intelligence where computers learn patterns from data without being explicitly programmed. There are three main types: supervised learning (learning from labeled examples), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through trial and error with rewards).",
    },
    {
        "prompt": "Explain the concept of overfitting.",
        "response": "Overfitting occurs when a machine learning model learns the training data too well, including its noise and outliers, rather than the underlying pattern. The model performs excellently on training data but poorly on new, unseen data. Common solutions include: using more training data, applying regularization (L1/L2), dropout, early stopping, or cross-validation.",
    },
    {
        "prompt": "What is the difference between classification and regression?",
        "response": "Classification predicts discrete categories (e.g., spam/not spam, cat/dog), while regression predicts continuous numerical values (e.g., house prices, temperature). Classification outputs a class label; regression outputs a number. Both are supervised learning tasks but use different evaluation metrics — accuracy for classification, MSE/MAE for regression.",
    },
    {
        "prompt": "How does gradient descent work?",
        "response": "Gradient descent is an optimization algorithm that minimizes a loss function by iteratively adjusting model parameters. It calculates the gradient (slope) of the loss with respect to each parameter, then updates parameters in the opposite direction of the gradient. The learning rate controls step size. Variants include batch, stochastic (SGD), and mini-batch gradient descent.",
    },
    {
        "prompt": "What is transfer learning?",
        "response": "Transfer learning is a technique where a model trained on one task is reused as the starting point for a different but related task. Instead of training from scratch, you take a pre-trained model (e.g., BERT, GPT, ResNet) and fine-tune it on your specific dataset. This saves training time, requires less data, and often achieves better results than training from scratch.",
    },
    {
        "prompt": "Explain what a neural network is.",
        "response": "A neural network is a computational model inspired by the human brain. It consists of layers of interconnected nodes (neurons): an input layer, one or more hidden layers, and an output layer. Each connection has a weight that is adjusted during training. Data flows forward through the network, and errors are propagated backward (backpropagation) to update weights.",
    },
    {
        "prompt": "What is cross-validation?",
        "response": "Cross-validation is a technique for evaluating model performance by splitting data into multiple folds. In k-fold CV, data is divided into k subsets. The model is trained on k-1 folds and tested on the remaining fold, rotating through all folds. This gives a more reliable performance estimate than a single train/test split and helps detect overfitting.",
    },
    {
        "prompt": "What is the bias-variance tradeoff?",
        "response": "The bias-variance tradeoff is a fundamental concept in ML. High bias means the model is too simple and underfits (misses patterns). High variance means the model is too complex and overfits (captures noise). The goal is to find the sweet spot — a model complex enough to capture the true pattern but simple enough to generalize to new data.",
    },
]

with open("teacher_responses.jsonl", "w") as f:
    for item in teacher_data:
        f.write(json.dumps(item) + "\n")

print(f"Created {len(teacher_data)} teacher-generated examples")
print("In production, these would come from GPT-4, Claude, or a local 70B model.")

In [ ]:
# Step 4: Create config for synthetic data generation + training
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 512
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  output_dir: "./synth_checkpoints"
  max_steps: 50
  per_device_train_batch_size: 4
  learning_rate: 2.0e-5

# Use the generated synthetic data for training
data:
  dataset_name_or_path: "synthetic_train.jsonl"  # will be created by --generate-data

# Synthetic data generation config
synthetic:
  enabled: true
  teacher_model: "n/a"               # not used for file backend
  teacher_backend: "file"            # read pre-generated responses
  seed_file: "teacher_responses.jsonl"
  output_file: "synthetic_train.jsonl"
  output_format: "messages"           # chat format for SFT training
  system_prompt: "You are a helpful AI tutor specializing in machine learning."
"""

with open("synth_config.yaml", "w") as f:
    f.write(config_yaml)
print("Config saved!")
print("  Backend: file (pre-generated responses)")
print("  Output format: messages (chat)")

In [ ]:
# Step 5: Generate synthetic training data
!forgelm --config synth_config.yaml --generate-data

In [ ]:
# Step 6: Inspect generated data
import json
import os

if os.path.isfile("synthetic_train.jsonl"):
    with open("synthetic_train.jsonl") as f:
        entries = [json.loads(line) for line in f]
    print(f"Generated {len(entries)} training examples\n")
    print("Sample entry:")
    print(json.dumps(entries[0], indent=2))
else:
    print("ERROR: synthetic_train.jsonl not created. Check --generate-data output above.")

In [ ]:
# Step 7: Train the student model on synthetic data
!forgelm --config synth_config.yaml

In [ ]:
# Step 8: Compare base vs synthetic-trained model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./synth_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: Model not found. Ensure training completed.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading synthetic-trained model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Test with UNSEEN ML questions (not in teacher data)
    test_prompts = [
        "What is a random forest?",
        "Explain the difference between bagging and boosting.",
        "What is dimensionality reduction?",
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[SYNTHETIC-TRAINED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("The synthetic-trained model should give more detailed, structured ML explanations")
    print("matching the teacher's style.")

## Using API Backend (Production)

For real synthetic data generation, use the API backend with a strong teacher model:

```yaml
synthetic:
  enabled: true
  teacher_model: "gpt-4o"
  teacher_backend: "api"
  api_base: "https://api.openai.com/v1"
  api_key_env: "OPENAI_API_KEY"  # read from env var (secure)
  seed_file: "my_prompts.txt"    # one prompt per line
  system_prompt: "You are an expert in machine learning. Give detailed, accurate answers."
  temperature: 0.7
  max_new_tokens: 1024
  output_file: "synthetic_data.jsonl"
  output_format: "messages"
```

Then run:
```bash
export OPENAI_API_KEY="your-key"
forgelm --config config.yaml --generate-data  # generate data
forgelm --config config.yaml                   # train on it
```

## Output Formats

| Format | Structure | Best For |
|--------|-----------|----------|
| `messages` | `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` | Chat/SFT training |
| `instruction` | `{"instruction": "...", "output": "..."}` | Alpaca-style fine-tuning |
| `chatml` | `{"User": "...", "Assistant": "..."}` | Legacy format |
| `prompt_response` | `{"prompt": "...", "response": "..."}` | Generic |

## Local Teacher (No API Needed)

Use a local HuggingFace model as the teacher:

```yaml
synthetic:
  enabled: true
  teacher_model: "meta-llama/Llama-3.1-8B-Instruct"
  teacher_backend: "local"  # loads model on GPU
  seed_file: "prompts.txt"
```

This is useful for air-gapped environments or when you want full control over the teacher.